In [ ]:
import pandas as pd
import numpy as np
import sys
import seaborn as sns
import matplotlib.pyplot as plt
sys.path.append("../")

import src.plots as pt
import src.features as ft

In [ ]:
dataset = pd.read_csv('../data/german_credit_data.csv')

In [ ]:
dataset

In [ ]:
dataset['Risk'] = dataset['Risk'].map({
    'good': 1,
    'bad': 0
})

In [ ]:
pt.plot_class_distribution(dataset, "Risk")

In [ ]:
dataset.isnull().sum()

In [ ]:
dataset = dataset.drop(columns=['Unnamed: 0','Saving accounts', 'Checking account'])

In [ ]:
dataset

In [ ]:
categoricas = ['Sex', 'Housing', 'Purpose', 'Risk']
dataset = pd.get_dummies(dataset, columns=categoricas, drop_first=True)

In [ ]:
bool_cols = dataset.select_dtypes(include='bool').columns
dataset[bool_cols] = dataset[bool_cols].astype(int)

In [ ]:
dataset

In [ ]:
dataset.info()

In [ ]:
dataset.info()

In [ ]:
dataset['Risk_1']

In [ ]:
corr_matrix = dataset.corr().abs()

upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

colunas_remover = []
print("Colunas com alta correlação (>= 0.7) e com quem estão mais correlacionadas:\n")
for col in upper.columns:
    correlacoes_altas = upper[col][upper[col] >= 0.7]
    if not correlacoes_altas.empty:
        mais_correlacionada = correlacoes_altas.idxmax()
        valor = correlacoes_altas.max()
        print(f"- {col} está altamente correlacionada com {mais_correlacionada} (correlação = {valor:.2f})")
        colunas_remover.append(col)

dataset.drop(columns=colunas_remover, inplace=True)

plt.figure(figsize=(70, 70))
sns.heatmap(dataset.corr(), annot=True, cmap='viridis')
plt.title("Matriz de Correlação (sem colunas altamente correlacionadas)")
plt.show()

In [ ]:
y = dataset['Risk_1']
X = dataset.drop(columns='Risk_1')
print(X.shape, y.shape)

In [ ]:
print(X.columns)

## CODECARBON

In [ ]:
porcentagens = [0, 10, 20, 30, 40, 50]
resultados_por_porcentagem = {}

for p in porcentagens:
    print(f"\n===== EXPERIMENTO COM {p}% DE REMOÇÃO =====", flush=True)
    
    dados_experimento = ft.reducao_codecarbon(p, X, y, 10)

    linhas = []
    for modelo, folds in dados_experimento.items():
        if not folds:
            continue
            
        for r in folds:
            linha = {
                "modelo": modelo,
                "fold": r["fold"],
                "acc": r["acc"],
                "prec": r["prec"],
                "rec": r["rec"],
                "f1": r["f1"],
                "emissions_g": r["emissions_g"],
                "energia_kwh": r["energia_kwh"],
                "tempo_s": r["tempo_s"],
                "rga": r["rga"],
                "emissions_grid_total_g": r["emissions_grid_total_g"],
                "energia_grid_total_kwh": r["energia_grid_total_kwh"],
                "params": str(r["params"])
            }
            linhas.append(linha)

    df_resultado = pd.DataFrame(linhas)
    
    df_resultado = df_resultado.sort_values(["modelo", "fold"])
    
    resultados_por_porcentagem[p] = df_resultado

    nome_saida = f"../reports/resultados/German/[REDUCAO {p}% - CC]-German.csv"

    df_resultado.to_csv(
        nome_saida,
        index=False,
        sep=';',
        encoding='utf-8-sig'
    )

    print(f"[SUCESSO] Experimento concluído! Arquivo gerado com sucesso: '{nome_saida}'", flush=True)

In [ ]:
porcentagens = [10,20,30,40, 50]
resultados_por_porcentagem = {}

for p in porcentagens:
    print(f"\n===== EXPERIMENTO COM {p}% DE AUMENTO =====", flush=True)
    
    dados_experimento = ft.aumento_codecarbon(p, X, y, 10,'german')

    linhas = []
    for modelo, folds in dados_experimento.items():
        if not folds:
            continue
            
        for r in folds:
            linha = {
                "modelo": modelo,
                "fold": r["fold"],
                "acc": r["acc"],
                "prec": r["prec"],
                "rec": r["rec"],
                "f1": r["f1"],
                "emissions_g": r["emissions_g"],
                "energia_kwh": r["energia_kwh"],
                "tempo_s": r["tempo_s"],
                "rga": r["rga"],
                "emissions_grid_total_g": r["emissions_grid_total_g"],
                "energia_grid_total_kwh": r["energia_grid_total_kwh"],
                "params": str(r["params"])
            }
            linhas.append(linha)

    df_resultado = pd.DataFrame(linhas)
    
    df_resultado = df_resultado.sort_values(["modelo", "fold"])
    
    resultados_por_porcentagem[p] = df_resultado

    nome_saida = f"../reports/resultados/German/[AUMENTO {p}% - CC]-German.csv"

    df_resultado.to_csv(
        nome_saida,
        index=False,
        sep=';',
        encoding='utf-8-sig'
    )

    print(f"[SUCESSO] Experimento concluído! Arquivo gerado com sucesso: '{nome_saida}'", flush=True)